# Virus micrograph classification (3-class local sample)

Transfer-learning experiments on the small sample of virus electron-microscopy images in this repo: **Adenovirus (14), Cowpox (10), Dengue (7)**, 31 images in total. Run this notebook from the repo root (the folder that contains `Adenovirus/`, `Cowpox/` and `Dengue/`).

What changed from the earlier version of this notebook:

1. **Independent test set.** The old "test" generator used the same 5 images as validation. Images are now split **once**, stratified by class, into train / validation / test, with a file-overlap check.
2. **Correct preprocessing per model.** Each backbone uses its own `preprocess_input` (the old notebook used MobileNetV2's for ResNet-50 and EfficientNet too).
3. **Clean evaluation.** Validation and test use no augmentation and `shuffle=False`, and every model is actually evaluated on the test set (the `evaluate_model` calls used to be commented out).
4. **More reliable estimate.** With only 31 images a single test split is tiny, so section 7 adds stratified 5-fold cross-validation of a linear probe on frozen MobileNetV2 features, where **every image is tested once**.
5. **Removed the UNet demo.** It was trained on random noise and produced no meaningful result.

The 22-class run on the larger dataset (Google Drive copy) is documented in the README; it came from an earlier version of this notebook that is kept in the git history.

In [ ]:
import os, random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.applications import mobilenet_v2, resnet50, efficientnet
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, Reshape, LSTM
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

BASE_DIR = '.'                                      # repo root
CLASSES = ['Adenovirus', 'Cowpox', 'Dengue']        # fixed order -> indices 0..2
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
IMG_EXT = ('.tif', '.tiff', '.png', '.jpg', '.jpeg')

## 1. Build the file table

Only the micrographs are used. The `particle_positions/` and `tags/` sub-folders hold `.txt` and `.csv` files that the notebook never reads.

In [ ]:
rows = []
for cls in CLASSES:
    folder = os.path.join(BASE_DIR, cls)
    for f in sorted(os.listdir(folder)):
        if f.lower().endswith(IMG_EXT):
            rows.append({'file_path': os.path.join(folder, f), 'labels': cls})
df = pd.DataFrame(rows)
assert len(df) > 0, 'No images found: run from the repo root'
print(df['labels'].value_counts().to_dict(), 'total', len(df))

## 2. Split once: train / validation / test

Stratified 60 / 20 / 20 on the image files, so the three sets are disjoint. The test set is only used in section 5.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.4, stratify=df['labels'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['labels'], random_state=SEED)

assert not (set(train_df.file_path) & set(val_df.file_path))
assert not (set(train_df.file_path) & set(test_df.file_path))
assert not (set(val_df.file_path) & set(test_df.file_path))

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f'{name:<5} {len(d):>2}', d['labels'].value_counts().to_dict())

## 3. Generators

Augmentation on training only. Each model gets its own `preprocess_input`.

In [ ]:
def make_generators(preprocess):
    aug = ImageDataGenerator(preprocessing_function=preprocess, rotation_range=20, zoom_range=0.2,
                             shear_range=0.2, width_shift_range=0.1, height_shift_range=0.1,
                             horizontal_flip=True)
    plain = ImageDataGenerator(preprocessing_function=preprocess)
    def flow(gen, frame, shuffle):
        return gen.flow_from_dataframe(frame, x_col='file_path', y_col='labels', classes=CLASSES,
                                       target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                       class_mode='categorical', shuffle=shuffle, seed=SEED)
    return flow(aug, train_df, True), flow(plain, val_df, False), flow(plain, test_df, False)

NUM_CLASSES = len(CLASSES)
early_stop = lambda: EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

## 4. Models

All are ImageNet-pretrained backbones with a small head, up to 30 epochs, early stopping on validation loss (patience 5, best weights restored).

In [ ]:
def build_mobilenet(finetune):
    base = MobileNetV2(include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,))
    base.trainable = finetune
    if finetune:
        for layer in base.layers[:-30]:
            layer.trainable = False
        head = [GlobalAveragePooling2D(), Dropout(0.5), Dense(128, activation='relu'), Dropout(0.3)]
        opt = Adam(learning_rate=1e-4)
    else:
        head = [GlobalAveragePooling2D(), Dropout(0.3)]
        opt = 'adam'
    m = Sequential([base] + head + [Dense(NUM_CLASSES, activation='softmax')])
    m.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return m

def build_resnet():
    base = ResNet50(include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,))
    base.trainable = False
    m = Sequential([base, GlobalAveragePooling2D(), Dropout(0.4), Dense(NUM_CLASSES, activation='softmax')])
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

def build_effnet_lstm():
    base = EfficientNetB0(include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,))
    base.trainable = False
    m = Sequential([base, Reshape((49, 1280)), LSTM(64), Dropout(0.3), Dense(NUM_CLASSES, activation='softmax')])
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

# name -> (builder, preprocess function). EfficientNet rescales inside the model.
EXPERIMENTS = {
    'MobileNetV2 (fine-tuned)': (lambda: build_mobilenet(True), mobilenet_v2.preprocess_input),
    'MobileNetV2 (frozen)':     (lambda: build_mobilenet(False), mobilenet_v2.preprocess_input),
    'ResNet-50 (frozen)':       (build_resnet, resnet50.preprocess_input),
    'EfficientNetB0 + LSTM':    (build_effnet_lstm, efficientnet.preprocess_input),
}

In [ ]:
results, histories = {}, {}
for name, (builder, preprocess) in EXPERIMENTS.items():
    tf.keras.backend.clear_session()
    train_ds, val_ds, test_ds = make_generators(preprocess)
    model = builder()
    h = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[early_stop()], verbose=0)
    best = int(np.argmin(h.history['val_loss']))
    histories[name] = h.history
    results[name] = (model, test_ds, val_ds)
    print(f"{name:<26} epochs run {len(h.history['loss']):>2}  best epoch {best + 1:>2}  "
          f"val loss {h.history['val_loss'][best]:.3f}  val acc {h.history['val_accuracy'][best]:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, h in histories.items():
    ax[0].plot(h['val_loss'], label=name); ax[1].plot(h['val_accuracy'], label=name)
ax[0].set_title('Validation loss'); ax[1].set_title('Validation accuracy')
for a in ax: a.set_xlabel('Epoch'); a.legend()
plt.tight_layout(); plt.show()

## 5. Test-set evaluation

The test set is tiny, so read the confidence interval, not just the point estimate. Models are listed for comparison; do not pick the best one from these few images.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half

def evaluate_clean(model, ds, name):
    probs = model.predict(ds, verbose=0)
    y_pred, y_true = probs.argmax(1), ds.classes
    k, n = int((y_pred == y_true).sum()), len(y_true)
    lo, hi = wilson(k, n)
    print(f'{name}: test accuracy {k}/{n} = {k / n:.3f}  (95% interval {lo:.2f}-{hi:.2f})')
    return y_true, y_pred

preds = {}
for name, (model, test_ds, _) in results.items():
    preds[name] = evaluate_clean(model, test_ds, name)

In [ ]:
y_true, y_pred = preds['MobileNetV2 (frozen)']
print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=CLASSES, zero_division=0))
sns.heatmap(confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES)), annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('MobileNetV2 (frozen): test confusion matrix'); plt.show()

## 6. Predict a single image

Uses the same preprocessing the model was trained with.

In [ ]:
def predict_image(img_path, model, preprocess):
    img = load_img(img_path, target_size=IMG_SIZE)
    x = preprocess(np.expand_dims(img_to_array(img), axis=0))
    pred = model.predict(x, verbose=0)[0]
    plt.imshow(img); plt.axis('off')
    plt.title(f'Predicted: {CLASSES[pred.argmax()]} ({pred.max() * 100:.1f}%)'); plt.show()

# example: predict_image(test_df.file_path.iloc[0], results['MobileNetV2 (frozen)'][0], mobilenet_v2.preprocess_input)

## 7. Cross-validated estimate (every image tested once)

A single 6-image test set cannot tell models apart. Here a **logistic-regression linear probe** is trained on frozen MobileNetV2 features (no augmentation) with stratified 5-fold cross-validation, so all 31 images are scored by a model that never saw them.

In [ ]:
def load_batch(paths):
    return mobilenet_v2.preprocess_input(np.stack([img_to_array(load_img(p, target_size=IMG_SIZE)) for p in paths]))

extractor = MobileNetV2(include_top=False, weights='imagenet', pooling='avg', input_shape=IMG_SIZE + (3,))
X = extractor.predict(load_batch(df.file_path), verbose=0)            # (n_images, 1280)
y = df['labels'].map({c: i for i, c in enumerate(CLASSES)}).values

oof = np.zeros(len(y), dtype=int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for tr, te in skf.split(X, y):
    clf = LogisticRegression(max_iter=2000).fit(X[tr], y[tr])
    oof[te] = clf.predict(X[te])

k, n = int((oof == y).sum()), len(y)
lo, hi = wilson(k, n)
print(f'5-fold cross-validated accuracy: {k}/{n} = {k / n:.3f}  (95% interval {lo:.2f}-{hi:.2f})')
print(classification_report(y, oof, target_names=CLASSES, zero_division=0))
sns.heatmap(confusion_matrix(y, oof), annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Linear probe, 5-fold out-of-fold predictions'); plt.show()